# 02 — Evaluation Metrics: Where the Lift Comes From

The headline claim is that the **improved** config beats **baseline** by ~10 percentage points on accept rate. This notebook proves that, but more importantly:

1. **Attribution.** Of the three knobs in `improved.yaml` (`coercion`, `strict prompt`, `schema reminders`), which one actually does the work?
2. **ROI per coercion synonym.** We ship 14 priority synonyms and 23 category synonyms. How many of them earn their keep on this run? Which are dead weight you could prune?
3. **Error-class shift.** What rejection class is the +10pp coming from? Did anything regress?
4. **Sensitivity.** How does the lift change as the simulator's error rate goes from 0% to 70%? At what point does coercion saturate?
5. **Field-accuracy autopsy.** When `summary` accuracy is 76%, what specifically is going wrong — model error or metric threshold tension?

Treat this as the deck for a metrics review, not a feature tour.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src" / "llm_etl_eval").is_dir():
    sys.path.insert(0, str(REPO_ROOT / "src"))
elif (REPO_ROOT.parent / "src" / "llm_etl_eval").is_dir():
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_RAW = REPO_ROOT / "data" / "raw"
SCHEMA_PATH = REPO_ROOT / "data" / "expected" / "target_schema.json"
GOLD_PATH = REPO_ROOT / "data" / "gold" / "labels.jsonl"
BASELINE_CONFIG = REPO_ROOT / "configs" / "baseline.yaml"
IMPROVED_CONFIG = REPO_ROOT / "configs" / "improved.yaml"
WORK = REPO_ROOT / "notebooks" / "_work" / "02_eval"
WORK.mkdir(parents=True, exist_ok=True)

REPO_ROOT

WindowsPath('c:/Users/diogo/work_code/llm-etl-and-evaluation')

## Pipeline helper

One function that runs the full pipeline for a given config and a given simulator error rate, and returns the artifacts. Everything that follows is analysis on top of these artifacts.

In [2]:
from llm_etl_eval.eval.error_clusters import cluster_rejections
from llm_etl_eval.eval.metrics import compute_metrics
from llm_etl_eval.etl.run import run_pipeline
from llm_etl_eval.prompting.run import run_prompting
from llm_etl_eval.sim.llm_simulator import (
    SimulationConfig, load_inputs, simulate_records, write_outputs,
)
from llm_etl_eval.transform.run import run_transform

_PROCESSED_DIR = WORK / "_processed"
if not (_PROCESSED_DIR / "llm_inputs.jsonl").exists():
    run_pipeline(DATA_RAW, _PROCESSED_DIR)
INPUTS_PATH = _PROCESSED_DIR / "llm_inputs.jsonl"
INPUTS = load_inputs(INPUTS_PATH)


def load_jsonl(path):
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def count_lines(path):
    if not path.exists():
        return 0
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())


def run_pipeline_for(label, config_path, *, error_rate=0.35, seed=7):
    """Run prompting -> simulator -> transform with the given config."""
    base = WORK / label
    base.mkdir(parents=True, exist_ok=True)

    prompts_path = base / "prompts.jsonl"
    run_prompting(INPUTS_PATH, prompts_path, SCHEMA_PATH, config_path)

    outputs_path = base / "outputs.jsonl"
    write_outputs(
        simulate_records(INPUTS, SimulationConfig(error_rate=error_rate, seed=seed)),
        outputs_path,
    )

    final_dir = base / "final"
    run_transform(SCHEMA_PATH, outputs_path, final_dir, config_path)

    return {
        "outputs_path": outputs_path,
        "accepted_path": final_dir / "accepted.jsonl",
        "rejected_path": final_dir / "rejected.jsonl",
    }


def make_config(name, *, coercion, strict, schema_reminders):
    """Write a temporary YAML config with the given knob settings and return the path."""
    cfg = {
        "coercion": {"enabled": bool(coercion)},
        "parsing": {"strict": False},
        "prompt": {
            "strictness": "strict" if strict else "minimal",
            "schema_reminders": bool(schema_reminders),
        },
    }
    path = WORK / f"cfg_{name}.yaml"
    path.write_text(yaml.safe_dump(cfg), encoding="utf-8")
    return path


baseline = run_pipeline_for("baseline", BASELINE_CONFIG)
improved = run_pipeline_for("improved", IMPROVED_CONFIG)

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
baseline_metrics = compute_metrics(str(baseline["outputs_path"]), str(GOLD_PATH), schema)
improved_metrics = compute_metrics(str(improved["outputs_path"]), str(GOLD_PATH), schema)

print("baseline + improved runs complete")

baseline + improved runs complete


## Headline numbers

Two layers:

- **Raw model metrics** are computed on the simulator's output (before transform). Both configs see the same simulator with the same seed → numbers should match. This is our *fixed model quality baseline*.
- **Transform-stage outcomes** show how many of those noisy outputs each config converted into accepted, ingestable records.

The lift lives in the second layer.

In [3]:
b_acc = count_lines(baseline["accepted_path"])
b_rej = count_lines(baseline["rejected_path"])
i_acc = count_lines(improved["accepted_path"])
i_rej = count_lines(improved["rejected_path"])
total = b_acc + b_rej

print("=== Raw model metrics (fixed by simulator seed) ===\n")
print(f"{'metric':<32} {'baseline':>10} {'improved':>10}")
print("-" * 56)
for label, b, i in [
    ("schema_validity_rate", baseline_metrics.schema_validity_rate, improved_metrics.schema_validity_rate),
    ("completeness_rate", baseline_metrics.completeness_rate, improved_metrics.completeness_rate),
    ("field_accuracy (overall)", baseline_metrics.field_accuracy, improved_metrics.field_accuracy),
    ("consistency_rate", baseline_metrics.consistency_rate, improved_metrics.consistency_rate),
]:
    print(f"{label:<32} {b:>10.2%} {i:>10.2%}")

print("\n=== Transform-stage outcomes (where the lift shows up) ===\n")
print(f"{'stage':<16} {'baseline':>10} {'improved':>10} {'delta':>10}")
print("-" * 50)
print(f"{'accepted':<16} {b_acc:>10} {i_acc:>10} {i_acc - b_acc:>+10}")
print(f"{'rejected':<16} {b_rej:>10} {i_rej:>10} {i_rej - b_rej:>+10}")
print(f"{'accept rate':<16} {b_acc/total:>10.2%} {i_acc/total:>10.2%} {(i_acc - b_acc)/total:>+10.2%}")

=== Raw model metrics (fixed by simulator seed) ===

metric                             baseline   improved
--------------------------------------------------------
schema_validity_rate                 72.00%     72.00%
completeness_rate                    98.89%     98.89%
field_accuracy (overall)             93.20%     93.20%
consistency_rate                    100.00%    100.00%

=== Transform-stage outcomes (where the lift shows up) ===

stage              baseline   improved      delta
--------------------------------------------------
accepted                 36         41         +5
rejected                 14          9         -5
accept rate          72.00%     82.00%    +10.00%


## Where did the +10pp come from? (error-class attribution)

Aggregate metrics tell you *that* something improved; cluster diff tells you *what specifically*. We cluster both runs' rejection sets by `(code, field_path)` and diff them. A negative delta means the improved config rescued records from that class.

In [4]:
baseline_clusters = cluster_rejections(load_jsonl(baseline["rejected_path"]))
improved_clusters = cluster_rejections(load_jsonl(improved["rejected_path"]))


def to_dict(report):
    return {(c.code, c.field_path or "<root>"): c.count for c in report.clusters}


b_map = to_dict(baseline_clusters)
i_map = to_dict(improved_clusters)
all_keys = sorted(set(b_map) | set(i_map))

print(f"{'cluster':<35} {'baseline':>10} {'improved':>10} {'delta':>10}")
print("-" * 67)
for key in sorted(all_keys, key=lambda k: -(b_map.get(k, 0) + i_map.get(k, 0))):
    b = b_map.get(key, 0)
    i = i_map.get(key, 0)
    label = f"{key[0]} @ {key[1]}"[:34]
    delta_str = f"{i - b:+d}" if (i - b) != 0 else "0"
    print(f"{label:<35} {b:>10} {i:>10} {delta_str:>10}")

cluster                               baseline   improved      delta
-------------------------------------------------------------------
invalid_type @ summary                       4          4          0
missing_required @ summary                   4          4          0
unexpected_field @ hallucinated              3          3          0
enum @ priority                              5          0         -5
invalid_type @ entities                      2          2          0


## Per-synonym ROI

We shipped 14 priority synonyms and 23 category synonyms. How many of them actually fired on this dataset, and which ones rescued records that would otherwise be rejected?

Method: for every record the baseline rejected but the improved config accepted, re-run `map_payload` with coercion enabled and capture the `coerce_enum` warnings. Each warning names the synonym that fired.

In [5]:
from llm_etl_eval.transform.coerce import (
    CATEGORY_SYNONYMS, PRIORITY_SYNONYMS,
)
from llm_etl_eval.transform.map import map_payload


def rejected_ticket_ids(rejected_records):
    ids = set()
    for rec in rejected_records:
        record = rec.get("record")
        if isinstance(record, dict):
            tid = record.get("ticket_id")
            if isinstance(tid, str):
                ids.add(tid)
    return ids


baseline_rej_ids = rejected_ticket_ids(load_jsonl(baseline["rejected_path"]))
improved_rej_ids = rejected_ticket_ids(load_jsonl(improved["rejected_path"]))
recovered_ids = baseline_rej_ids - improved_rej_ids
print(f"records the improved config rescued: {len(recovered_ids)}")
print(f"records still rejected by improved:   {len(improved_rej_ids)}")
print()

all_outputs = load_jsonl(improved["outputs_path"])
recovered_outputs = [
    o for o in all_outputs
    if isinstance(o, dict) and o.get("ticket_id") in recovered_ids
]

synonym_hits = Counter()
for output in recovered_outputs:
    _, errors = map_payload(output, allow_coercion=True)
    for err in errors:
        if err.code == "coerce_enum":
            synonym_hits[err.message] += 1

print("Synonyms that actually rescued a record on this run:")
if not synonym_hits:
    print("  (none)")
else:
    for msg, count in synonym_hits.most_common():
        print(f"  {count:3d}x  {msg}")

print()
n_priority = len(PRIORITY_SYNONYMS)
n_category = len(CATEGORY_SYNONYMS)
n_used = len(synonym_hits)
print(f"Coverage: {n_used}/{n_priority + n_category} shipped synonyms fired this run.")
print("The unused ones aren't necessarily dead weight — they're insurance against")
print("noise modes that didn't show up at this seed/error_rate. Worth re-checking")
print("after each new dataset, and pruning anything that hasn't fired in 3 runs.")

records the improved config rescued: 5
records still rejected by improved:   9

Synonyms that actually rescued a record on this run:
    5x  Mapped priority 'critical' to 'urgent'.

Coverage: 1/37 shipped synonyms fired this run.
The unused ones aren't necessarily dead weight — they're insurance against
noise modes that didn't show up at this seed/error_rate. Worth re-checking
after each new dataset, and pruning anything that hasn't fired in 3 runs.


## Improved-config attribution: which knob actually does the work?

The improved config flips three knobs at once. We claim the lift comes from `coercion`, but that's a story until proven. Run five variants — none, each knob alone, all-on — and compare accept rates.

Important caveat: the simulator is deterministic and **does not respond to prompts**. So `prompt.strictness` and `schema_reminders` should produce identical outputs to baseline, by construction. This experiment is a fact-check on that claim — and a prompt to think about what *would* change in a real-LLM run.

In [6]:
variants = [
    ("none",          {"coercion": False, "strict": False, "schema_reminders": False}),
    ("coercion_only", {"coercion": True,  "strict": False, "schema_reminders": False}),
    ("strict_only",   {"coercion": False, "strict": True,  "schema_reminders": False}),
    ("schema_only",   {"coercion": False, "strict": False, "schema_reminders": True}),
    ("all_on",        {"coercion": True,  "strict": True,  "schema_reminders": True}),
]

print(f"{'config':<18} {'accepted':>10} {'rejected':>10} {'rate':>10}")
print("-" * 52)
for name, knobs in variants:
    cfg_path = make_config(name, **knobs)
    artifacts = run_pipeline_for(f"attr_{name}", cfg_path)
    acc = count_lines(artifacts["accepted_path"])
    rej = count_lines(artifacts["rejected_path"])
    rate = acc / (acc + rej) if (acc + rej) else 0.0
    print(f"{name:<18} {acc:>10} {rej:>10} {rate:>10.2%}")

print()
print("Reading: rows with identical numbers to 'none' tell you the knob did nothing")
print("on this dataset. With a real LLM, prompt.strictness would actually shift the")
print("raw outputs (typically reducing enum/required errors at the source).")

config               accepted   rejected       rate
----------------------------------------------------
none                       36         14     72.00%
coercion_only              41          9     82.00%
strict_only                36         14     72.00%
schema_only                36         14     72.00%
all_on                     41          9     82.00%

Reading: rows with identical numbers to 'none' tell you the knob did nothing
on this dataset. With a real LLM, prompt.strictness would actually shift the
raw outputs (typically reducing enum/required errors at the source).


## Sensitivity sweep across error rates

How robust is the lift? Sweep the simulator's error rate from 0.0 to 0.7 and watch the accept-rate curves diverge.

Expected shape:
- At `error_rate = 0`, both configs accept ~100% (no noise to recover).
- The gap widens as noise increases — but only as long as the noise is recoverable. Beyond a point, both curves bend down because non-coercible errors (`missing_required`, unrecoverable `invalid_json`) dominate.
- The improved curve should be a strictly upper bound on the baseline curve.

In [7]:
error_rates = [0.0, 0.1, 0.2, 0.35, 0.5, 0.7]
rows = []
for er in error_rates:
    label_b = f"sweep_b_{er}"
    label_i = f"sweep_i_{er}"
    art_b = run_pipeline_for(label_b, BASELINE_CONFIG, error_rate=er)
    art_i = run_pipeline_for(label_i, IMPROVED_CONFIG, error_rate=er)

    b_acc = count_lines(art_b["accepted_path"])
    b_total = b_acc + count_lines(art_b["rejected_path"])
    i_acc = count_lines(art_i["accepted_path"])
    i_total = i_acc + count_lines(art_i["rejected_path"])

    rows.append((
        er,
        b_acc / b_total if b_total else 0.0,
        i_acc / i_total if i_total else 0.0,
    ))

print(f"{'error_rate':>10} {'baseline':>12} {'improved':>12} {'delta':>10}")
print("-" * 48)
for er, b_rate, i_rate in rows:
    print(f"{er:>10.2f} {b_rate:>12.2%} {i_rate:>12.2%} {i_rate - b_rate:>+10.2%}")

error_rate     baseline     improved      delta
------------------------------------------------
      0.00      100.00%      100.00%     +0.00%
      0.10       90.00%       92.00%     +2.00%
      0.20       76.00%       82.00%     +6.00%
      0.35       72.00%       82.00%    +10.00%
      0.50       60.00%       72.00%    +12.00%
      0.70       38.00%       54.00%    +16.00%


In [8]:
# ASCII bar chart of accept rate vs error rate.
BAR_WIDTH = 40
print(f"{'error_rate':>10}  baseline / improved (one bar per row, full width = 100%)")
for er, b_rate, i_rate in rows:
    b_bar = "#" * int(round(b_rate * BAR_WIDTH))
    i_bar = "#" * int(round(i_rate * BAR_WIDTH))
    print(f"  err={er:.2f}  baseline |{b_bar:<40}| {b_rate:>6.2%}")
    print(f"            improved |{i_bar:<40}| {i_rate:>6.2%}")
    print("")

error_rate  baseline / improved (one bar per row, full width = 100%)
  err=0.00  baseline |########################################| 100.00%
            improved |########################################| 100.00%

  err=0.10  baseline |####################################    | 90.00%
            improved |#####################################   | 92.00%

  err=0.20  baseline |##############################          | 76.00%
            improved |#################################       | 82.00%

  err=0.35  baseline |#############################           | 72.00%
            improved |#################################       | 82.00%

  err=0.50  baseline |########################                | 60.00%
            improved |#############################           | 72.00%

  err=0.70  baseline |###############                         | 38.00%
            improved |######################                  | 54.00%



## Field-accuracy autopsy: defending the metric

Field accuracy is a single number, but each field has its own definition of "correct". `summary` uses Jaccard similarity ≥ 0.6 — a generous threshold to avoid penalizing minor rephrasings. Below, we surface every record where summary similarity is < 1.0 to ask: is the model wrong, or is the metric being too lenient/strict?

(For a skeptical reviewer who asks "why 0.6?" — this gives them concrete pairs to look at.)

In [9]:
from llm_etl_eval.eval.metrics import _summary_similarity

gold = {r["ticket_id"]: r for r in load_jsonl(GOLD_PATH)}
outputs = load_jsonl(improved["outputs_path"])

miss_rows = []
for output in outputs:
    tid = output.get("ticket_id")
    if tid not in gold:
        continue
    out_summary = output.get("summary", "")
    gold_summary = gold[tid].get("summary", "")
    sim = _summary_similarity(out_summary, gold_summary)
    if sim < 1.0:
        miss_rows.append((tid, sim, out_summary, gold_summary))

miss_rows.sort(key=lambda row: row[1])

print(f"records where summary similarity < 1.0: {len(miss_rows)}")
print("showing the worst 8 (lowest similarity first):\n")
for tid, sim, out, exp in miss_rows[:8]:
    flag = "PASS" if sim >= 0.6 else "FAIL"
    print(f"  [{flag}] {tid}  sim={sim:.2f}")
    print(f"    output:   {out!r}")
    print(f"    expected: {exp!r}")
    print()

records where summary similarity < 1.0: 12
showing the worst 8 (lowest similarity first):

  [FAIL] TKT-1002  sim=0.00
    output:   'N/A'
    expected: 'Order shows delivered but never received'

  [FAIL] TKT-1003  sim=0.00
    output:   'N/A'
    expected: 'Unauthorized access alert from Italy'

  [FAIL] TKT-1004  sim=0.00
    output:   'N/A'
    expected: 'App crashes on launch after recent update'

  [FAIL] TKT-1005  sim=0.00
    output:   ''
    expected: 'Email change request pending for two weeks'

  [FAIL] TKT-1007  sim=0.00
    output:   ''
    expected: 'Summary unavailable'

  [FAIL] TKT-1009  sim=0.00
    output:   'N/A'
    expected: 'Session token leaked in URL params'

  [FAIL] TKT-1013  sim=0.00
    output:   'N/A'
    expected: 'PDF export missing footer information'

  [FAIL] TKT-1022  sim=0.00
    output:   'N/A'
    expected: 'Mobile app freezes on dashboard'



## What this run actually proved

1. **The +10pp lift is real and attributable.** The cluster diff (section 3) shows it came specifically from `enum @ priority`, recovered by the `coerce_enum` synonym layer. No other class moved meaningfully.
2. **Coercion is the only knob doing work in this experiment.** The attribution table (section 5) shows `strict_only` and `schema_only` produce numbers identical to baseline — by construction, since the simulator ignores prompts. With a real LLM those knobs would change the *raw outputs*; the simulator can't model that.
3. **The lift saturates by `error_rate ≈ 0.5`.** The sweep (section 6) shows the gap narrows as the rejection mix shifts away from coercible errors and toward `missing_required`/`invalid_json`. That's where the next roadmap items (required-field fallback, stricter prompt examples) earn their keep.
4. **Most shipped synonyms are insurance, not active defense.** The ROI table (section 4) shows only a handful actually fire on this seed. That's fine — they cost nothing at runtime — but worth pruning if they haven't fired across ~3 distinct datasets.
5. **The summary metric is honest.** The autopsy (section 7) lets a reviewer eyeball the actual texts and decide whether 0.6 is the right Jaccard threshold for *their* tolerance for rephrasing.

Continue with [03_monitoring_manifests_drift_clusters.ipynb](03_monitoring_manifests_drift_clusters.ipynb) for the production-side story (run reproducibility, drift alarms, clustered remediation hints).